# 🎬 OpenSource Clipping Studio — Google Colab (Fixed & Ready)

Notebook ini disusun untuk **Google Colab + GPU** dan menjalankan backend **OpenSource Clipping Studio** melalui **ngrok**.

### v4 CORS + API token fix

Versi ini:
- memuat `OSC_ALLOWED_ORIGINS` dari Colab Secrets dan meneruskannya ke subprocess FastAPI,
- memakai origin GitHub Pages `https://raymanq.github.io` sebagai fallback aman jika secret belum dibuat,
- memuat `OSC_API_TOKEN` tanpa pernah menampilkan nilainya,
- menyimpan konfigurasi tersebut ke environment dan `.env`,
- melakukan health check lokal dan health check publik melalui ngrok,
- memverifikasi header CORS sebelum menampilkan backend sebagai siap,
- tetap menjalankan FastAPI/Uvicorn di **subprocess Python bersih** agar NumPy/SciPy tidak tercampur dengan state kernel Colab.

### Pixabay B-roll support included

Notebook ini menambahkan **Pixabay Video API** sebagai sumber B-roll tanpa mengubah frontend Web Studio. Mode `auto` akan memakai Pixabay bila `PIXABAY_API_KEY` tersedia, lalu tetap kompatibel dengan alur B-roll Pexels milik proyek.

### Compatibility fix included

Notebook ini juga memasang **cuBLAS CUDA 12** dan **cuDNN 9** melalui paket NVIDIA Python, lalu menambahkan direktori library ke `LD_LIBRARY_PATH` sebelum backend dijalankan. Ini mencegah error `libcublas.so.12 is not found` saat Faster-Whisper mulai inference.

Notebook ini mem-pin **PyAV 18.1.0** karena Faster-Whisper 1.2.1 masih memakai argumen `metadata_errors` yang dihapus pada PyAV 19.0.0.

### Sebelum klik **Runtime → Run all**

1. Aktifkan **T4 GPU**: `Runtime → Change runtime type → T4 GPU`.
2. Tambahkan Colab Secrets berikut dan aktifkan **Notebook access**:
   - `GOOGLE_API_KEY` — **wajib**
   - `NGROK_AUTHTOKEN` — **wajib**
   - `OSC_ALLOWED_ORIGINS` — direkomendasikan; isi `https://raymanq.github.io`
   - `OSC_API_TOKEN` — direkomendasikan untuk melindungi endpoint backend
   - `PIXABAY_API_KEY` — opsional, **direkomendasikan untuk B-roll**
   - `PEXELS_API_KEY` — opsional, fallback B-roll jika ingin tetap memakai Pexels
   - `BROLL_PROVIDER` — opsional: `auto` (default), `pixabay`, atau `pexels`
   - `HF_TOKEN` — opsional, direkomendasikan untuk Hugging Face / diarization
   - `NVIDIA_API_KEY` — opsional, hanya jika memakai NVIDIA AI provider
3. Jika belum mempunyai `OSC_API_TOKEN`, buat di PowerShell:
   `python -c "import secrets; print(secrets.token_urlsafe(32))"`
4. Siapkan `youtube_cookies.txt` hasil export manual dalam format **Netscape cookies**.  
   Saat **Run all**, notebook akan meminta file ini jika belum tersedia.

> **Keamanan:** `youtube_cookies.txt`, `OSC_API_TOKEN`, dan seluruh API key adalah credential sensitif. Jangan upload ke GitHub, jangan dibagikan, dan jangan tampilkan isinya.

Setelah semua pemeriksaan lolos, cell terakhir akan menampilkan **Public URL ngrok**. Copy URL tersebut ke tombol **Connect** di OpenSource Clipping Studio.


## 1. Konfigurasi

In [ ]:
print("=== OPENSOURCE CLIPPING COLAB FIX V5 ===")
from pathlib import Path
from urllib.parse import urlsplit

REPO_URL = "https://github.com/RaymanQ/opensource-clipping.git"
PROJECT_DIR = Path("/content/opensource-clipping")

COOKIE_PATH = Path("/content/youtube_cookies.txt")
REQUIRE_YOUTUBE_COOKIES = True

WHISPER_SMOKE_MODEL = "tiny"
WHISPER_COMPUTE_TYPE = "float16"

BACKEND_PORT = 8000
STUDIO_URL = "https://raymanq.github.io/opensource-clipping/studio/"

_studio_parts = urlsplit(STUDIO_URL)
STUDIO_ORIGIN = f"{_studio_parts.scheme}://{_studio_parts.netloc}"

print("✅ Configuration loaded")
print(f"   Studio URL    : {STUDIO_URL}")
print(f"   Studio Origin : {STUDIO_ORIGIN}")


## 2. Setup Colab, Clone Repo, FFmpeg, Deno, dan Dependencies

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

def run(cmd, *, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Notebook ini dibuat untuk Google Colab. "
        "Buka file .ipynb ini di Colab lalu jalankan kembali."
    ) from exc

print("Python:", sys.version.split()[0])

gpu_check = subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU tidak terdeteksi. Pilih Runtime → Change runtime type → T4 GPU, "
        "lalu jalankan notebook dari awal."
    )

print("✅ NVIDIA GPU detected")

run(["apt-get", "update", "-qq"])
run([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg",
    "curl",
    "git",
    "libgl1",
    "libgles2",
    "libegl1",
    "libglib2.0-0",
    "libsm6",
    "libxext6",
    "libxrender1",
])

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])

requirements = PROJECT_DIR / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(f"requirements.txt tidak ditemukan: {requirements}")

req_text = requirements.read_text(encoding="utf-8")
if sys.version_info >= (3, 13) and "numpy<2.0.0" in req_text:
    req_text = req_text.replace(
        "numpy<2.0.0",
        'numpy>=2.1,<2.3; python_version >= "3.13"\n'
        'numpy<2.0.0; python_version < "3.13"'
    )
    requirements.write_text(req_text, encoding="utf-8")
    print("✅ Legacy NumPy pin disesuaikan untuk Python 3.13+")

deno_bin = Path("/root/.deno/bin/deno")

if not deno_bin.exists():
    run(["bash", "-lc", "curl -fsSL https://deno.land/install.sh | sh"])

os.environ["DENO_INSTALL"] = "/root/.deno"
os.environ["PATH"] = "/root/.deno/bin:" + os.environ.get("PATH", "")

run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-r", str(requirements)])

run([
    sys.executable, "-m", "pip", "install", "-U",
    "yt-dlp[default]",
    "faster-whisper==1.2.1",
    "ctranslate2==4.8.2",
    "av==18.1.0",
    "mediapipe==1.0.1",
    "pyngrok",
    "nest-asyncio",
    "aiofiles",
    "gdown",
])

# Faster-Whisper/CTranslate2 GPU inference needs the CUDA 12 cuBLAS
# and cuDNN shared libraries. Colab can expose a newer CUDA stack through
# PyTorch while libcublas.so.12 is still absent from the dynamic loader.
# Install the exact runtime libraries recommended by Faster-Whisper.
run([
    sys.executable, "-m", "pip", "install", "-U",
    "nvidia-cublas-cu12",
    "nvidia-cudnn-cu12==9.*",
])

# Resolve the pip-installed NVIDIA library directories WITHOUT importing
# nvidia.cublas.lib / nvidia.cudnn.lib. Those "lib" directories contain
# shared libraries and are not guaranteed to behave like normal Python
# modules on every Colab/Python combination.
from importlib.metadata import distribution

def nvidia_lib_dir(distribution_name, relative_path, required_glob):
    dist = distribution(distribution_name)
    lib_dir = Path(dist.locate_file(relative_path)).resolve()

    if not lib_dir.is_dir():
        raise RuntimeError(
            f"Direktori library {distribution_name} tidak ditemukan: {lib_dir}"
        )

    matches = sorted(lib_dir.glob(required_glob))
    if not matches:
        raise RuntimeError(
            f"Library {required_glob} tidak ditemukan di {lib_dir}"
        )

    return lib_dir

cublas_dir = nvidia_lib_dir(
    "nvidia-cublas-cu12",
    "nvidia/cublas/lib",
    "libcublas.so*",
)
cudnn_dir = nvidia_lib_dir(
    "nvidia-cudnn-cu12",
    "nvidia/cudnn/lib",
    "libcudnn.so*",
)

cuda_dirs = [str(cublas_dir), str(cudnn_dir)]
existing_ld = os.environ.get("LD_LIBRARY_PATH", "").strip()

# Keep order stable and avoid duplicate entries.
ld_entries = []
for entry in cuda_dirs + ([existing_ld] if existing_ld else []):
    for part in entry.split(":"):
        part = part.strip()
        if part and part not in ld_entries:
            ld_entries.append(part)

os.environ["LD_LIBRARY_PATH"] = ":".join(ld_entries)

print("✅ CUDA 12 runtime libraries installed")
print("   cuBLAS:", cublas_dir)
print("   cuDNN :", cudnn_dir)
print("✅ LD_LIBRARY_PATH configured for Faster-Whisper/CTranslate2")

# Important: changing LD_LIBRARY_PATH inside the already-running Colab kernel
# does not reliably change the loader search path of that same process.
# The smoke test and backend are launched as fresh subprocesses below, so
# they inherit this corrected LD_LIBRARY_PATH before importing CTranslate2.

# TensorFlow is not required by OpenSource Clipping.
# MediaPipe only imports it opportunistically for API-doc controls.
# Colab's preinstalled TensorFlow can conflict with the NumPy/SciPy stack,
# so remove it from this dedicated runtime before MediaPipe is imported.
print("🧹 Removing optional TensorFlow packages from OSC runtime...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "-y",
        "tensorflow",
        "tensorflow-cpu",
        "tensorflow-gpu",
        "tf-nightly",
        "tf-nightly-cpu",
    ],
    check=False,
)

for folder in ("uploads", "outputs", "custom_fonts"):
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.chdir(PROJECT_DIR)

run(["ffmpeg", "-version"])
run([str(deno_bin), "--version"])
run([sys.executable, "-m", "yt_dlp", "--version"])

print()
print("✅ Base setup complete")


## 3. Colab Secrets

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlsplit
from google.colab import userdata

def secret(name, default=""):
    try:
        value = userdata.get(name)
        return value.strip() if isinstance(value, str) else (value or default)
    except Exception:
        return default

GOOGLE_API_KEY = secret("GOOGLE_API_KEY")
NGROK_AUTHTOKEN = secret("NGROK_AUTHTOKEN")

# Web Studio / backend security
OSC_ALLOWED_ORIGINS = secret("OSC_ALLOWED_ORIGINS", STUDIO_ORIGIN).strip()
OSC_API_TOKEN = secret("OSC_API_TOKEN")

# Normalize and validate the CORS origin.
# An origin is scheme + host (+ optional port), never a path such as /studio/.
_origin_parts = urlsplit(OSC_ALLOWED_ORIGINS)
if (
    _origin_parts.scheme not in {"http", "https"}
    or not _origin_parts.netloc
    or _origin_parts.query
    or _origin_parts.fragment
    or _origin_parts.path not in {"", "/"}
):
    raise RuntimeError(
        "OSC_ALLOWED_ORIGINS tidak valid. Gunakan origin tanpa path, contoh: "
        "https://raymanq.github.io"
    )

OSC_ALLOWED_ORIGINS = f"{_origin_parts.scheme}://{_origin_parts.netloc}"

if OSC_ALLOWED_ORIGINS != STUDIO_ORIGIN:
    print(
        "⚠️ OSC_ALLOWED_ORIGINS berbeda dengan origin STUDIO_URL. "
        f"Configured={OSC_ALLOWED_ORIGINS} | Studio={STUDIO_ORIGIN}"
    )

# B-roll providers
PIXABAY_API_KEY = secret("PIXABAY_API_KEY")
PEXELS_API_KEY = secret("PEXELS_API_KEY")
BROLL_PROVIDER = secret("BROLL_PROVIDER", "auto").strip().lower()

HF_TOKEN = secret("HF_TOKEN")
NVIDIA_API_KEY = secret("NVIDIA_API_KEY")

missing = []
if not GOOGLE_API_KEY:
    missing.append("GOOGLE_API_KEY")
if not NGROK_AUTHTOKEN:
    missing.append("NGROK_AUTHTOKEN")

if missing:
    raise RuntimeError(
        "Colab Secrets belum lengkap: "
        + ", ".join(missing)
        + ". Tambahkan secret di sidebar 🔑 Secrets dan aktifkan Notebook access."
    )

if BROLL_PROVIDER not in {"auto", "pixabay", "pexels"}:
    raise RuntimeError(
        "BROLL_PROVIDER harus salah satu dari: auto, pixabay, pexels."
    )

# Resolve provider automatically.
if BROLL_PROVIDER == "auto":
    if PIXABAY_API_KEY:
        BROLL_PROVIDER = "pixabay"
    elif PEXELS_API_KEY:
        BROLL_PROVIDER = "pexels"
    else:
        BROLL_PROVIDER = "pexels"

if BROLL_PROVIDER == "pixabay" and not PIXABAY_API_KEY:
    raise RuntimeError(
        "BROLL_PROVIDER=pixabay tetapi PIXABAY_API_KEY belum diisi."
    )

if BROLL_PROVIDER == "pexels" and not PEXELS_API_KEY:
    print("⚠️ PEXELS_API_KEY tidak tersedia. B-roll Pexels mungkin dinonaktifkan.")

# Export everything required by the backend subprocess.
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["APP_TIMEZONE"] = "Asia/Jakarta"
os.environ["BROLL_PROVIDER"] = BROLL_PROVIDER
os.environ["OSC_ALLOWED_ORIGINS"] = OSC_ALLOWED_ORIGINS

if OSC_API_TOKEN:
    os.environ["OSC_API_TOKEN"] = OSC_API_TOKEN
else:
    os.environ.pop("OSC_API_TOKEN", None)

if PIXABAY_API_KEY:
    os.environ["PIXABAY_API_KEY"] = PIXABAY_API_KEY

# Existing OpenSource Clipping code checks PEXELS_API_KEY before using B-roll.
# When Pixabay is selected and no real Pexels key exists, this compatibility
# marker keeps the original B-roll path enabled. HTTP calls are rerouted by
# the Pixabay adapter cell below.
PEXELS_COMPAT_KEY = PEXELS_API_KEY
if BROLL_PROVIDER == "pixabay" and not PEXELS_COMPAT_KEY:
    PEXELS_COMPAT_KEY = "PIXABAY_PROXY_ACTIVE"

if PEXELS_COMPAT_KEY:
    os.environ["PEXELS_API_KEY"] = PEXELS_COMPAT_KEY

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

env_values = {
    "GOOGLE_API_KEY": GOOGLE_API_KEY,
    "PIXABAY_API_KEY": PIXABAY_API_KEY,
    "PEXELS_API_KEY": PEXELS_COMPAT_KEY,
    "BROLL_PROVIDER": BROLL_PROVIDER,
    "HF_TOKEN": HF_TOKEN,
    "NVIDIA_API_KEY": NVIDIA_API_KEY,
    "OSC_ALLOWED_ORIGINS": OSC_ALLOWED_ORIGINS,
    "OSC_API_TOKEN": OSC_API_TOKEN,
    "APP_TIMEZONE": "Asia/Jakarta",
}

env_text = "\n".join(
    f"{key}={value}"
    for key, value in env_values.items()
    if value
) + "\n"

(PROJECT_DIR / ".env").write_text(env_text, encoding="utf-8")

print("✅ Colab Secrets loaded")
print("  GOOGLE_API_KEY      : ✅ Set")
print("  NGROK_AUTHTOKEN     : ✅ Set")
print(f"  OSC_ALLOWED_ORIGINS : {OSC_ALLOWED_ORIGINS}")
print(f"  OSC_API_TOKEN       : {'✅ Set (hidden)' if OSC_API_TOKEN else '⚪ Not set'}")
print(f"  PIXABAY_API_KEY     : {'✅ Set' if PIXABAY_API_KEY else '⚪ Optional / not set'}")
print(f"  PEXELS_API_KEY      : {'✅ Set' if PEXELS_API_KEY else '⚪ Optional / not set'}")
print(f"  BROLL_PROVIDER      : {BROLL_PROVIDER}")
print(f"  HF_TOKEN            : {'✅ Set' if HF_TOKEN else '⚪ Optional / not set'}")
print(f"  NVIDIA_API_KEY      : {'✅ Set' if NVIDIA_API_KEY else '⚪ Optional / not set'}")


## 4. Pixabay API Adapter untuk B-roll

OpenSource Clipping saat ini memanggil format API Pexels untuk B-roll. Cell ini memasang adapter **khusus request video Pexels** dan launcher backend terpatch agar request dapat diarahkan ke **Pixabay Video API** tanpa mengubah Web Studio atau pipeline render.

- `BROLL_PROVIDER=auto` → Pixabay dipilih jika `PIXABAY_API_KEY` tersedia.
- `BROLL_PROVIDER=pixabay` → paksa Pixabay.
- `BROLL_PROVIDER=pexels` → gunakan Pexels asli.
- Jika Pixabay dipilih, hasil Pixabay dikonversi ke struktur `videos/video_files` yang sudah dipahami pipeline B-roll proyek.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

adapter_file = PROJECT_DIR / "osc_pixabay_adapter.py"
launcher_file = PROJECT_DIR / "osc_server.py"

adapter_source = r"""
# OSC_PIXABAY_BROLL_ADAPTER
# Transparently adapts Pixabay Video API responses to the Pexels-shaped
# payload expected by OpenSource Clipping's existing B-roll pipeline.
# Pixabay search responses are cached for 24 hours.

import hashlib
import json
import os
import time
from pathlib import Path
from urllib.parse import parse_qs, urlsplit

import requests


_CACHE_TTL_SECONDS = 24 * 60 * 60


def _pixabay_payload_to_pexels(payload):
    hits = payload.get("hits") or []
    converted = []

    for hit in hits:
        variants = hit.get("videos") or {}
        video_files = []
        video_pictures = []
        seen_urls = set()
        seen_thumbs = set()

        # Pixabay exposes large/medium/small/tiny variants.
        # Map them into the Pexels video_files schema already consumed
        # by OpenSource Clipping.
        for variant_name, quality in (
            ("large", "hd"),
            ("medium", "hd"),
            ("small", "sd"),
            ("tiny", "sd"),
        ):
            item = variants.get(variant_name) or {}
            link = item.get("url")
            thumbnail = item.get("thumbnail")

            if thumbnail and thumbnail not in seen_thumbs:
                seen_thumbs.add(thumbnail)
                video_pictures.append(
                    {
                        "id": f"{hit.get('id', '')}-{variant_name}-thumb",
                        "picture": thumbnail,
                        "nr": len(video_pictures),
                    }
                )

            if not link or link in seen_urls:
                continue

            seen_urls.add(link)
            video_files.append(
                {
                    "id": f"{hit.get('id', '')}-{variant_name}",
                    "quality": quality,
                    "file_type": "video/mp4",
                    "width": int(item.get("width") or 0),
                    "height": int(item.get("height") or 0),
                    "link": link,
                    "size": int(item.get("size") or 0),
                }
            )

        if not video_files:
            continue

        largest = max(
            video_files,
            key=lambda f: (f.get("width", 0) * f.get("height", 0)),
        )
        cover = video_pictures[0]["picture"] if video_pictures else ""

        converted.append(
            {
                "id": hit.get("id"),
                "width": largest.get("width", 0),
                "height": largest.get("height", 0),
                "duration": int(hit.get("duration") or 0),
                "url": hit.get("pageURL") or "https://pixabay.com/videos/",
                "image": cover,
                "user": hit.get("user") or "Pixabay",
                "video_files": video_files,
                "video_pictures": video_pictures,
                # Extra source metadata; safely ignored by the renderer.
                "_source": "pixabay",
                "_tags": hit.get("tags") or "",
            }
        )

    return {
        "page": 1,
        "per_page": len(converted),
        "total_results": int(payload.get("totalHits") or len(converted)),
        "url": "https://pixabay.com/videos/",
        "videos": converted,
    }


def _cache_path(params):
    cache_dir = Path(
        os.environ.get(
            "PIXABAY_CACHE_DIR",
            ".cache/pixabay_broll",
        )
    )
    cache_dir.mkdir(parents=True, exist_ok=True)

    safe_params = {
        key: value
        for key, value in params.items()
        if key != "key"
    }
    digest = hashlib.sha256(
        json.dumps(
            safe_params,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()

    return cache_dir / f"{digest}.json"


def _load_cached_payload(path):
    try:
        age = time.time() - path.stat().st_mtime
        if age > _CACHE_TTL_SECONDS:
            return None

        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict) and "videos" in payload:
            return payload
    except Exception:
        return None

    return None


def _save_cached_payload(path, payload):
    try:
        path.write_text(
            json.dumps(payload, ensure_ascii=False),
            encoding="utf-8",
        )
    except Exception:
        # Cache failure should never break video generation.
        pass


def _synthetic_json_response(payload, url):
    response = requests.Response()
    response.status_code = 200
    response.url = url
    response.encoding = "utf-8"
    response.headers["Content-Type"] = "application/json"
    response.headers["X-OSC-BRoll-Cache"] = "HIT"
    response._content = json.dumps(
        payload,
        ensure_ascii=False,
    ).encode("utf-8")
    return response


_original_request = requests.sessions.Session.request

if not getattr(_original_request, "_osc_pixabay_adapter", False):
    _printed_once = {"value": False}

    def _request_with_pixabay(self, method, url, **kwargs):
        provider = os.environ.get("BROLL_PROVIDER", "pexels").strip().lower()
        url_text = str(url)

        is_pexels_video_search = (
            "api.pexels.com/videos/search" in url_text
        )

        if provider == "pixabay" and is_pexels_video_search:
            api_key = os.environ.get("PIXABAY_API_KEY", "").strip()
            if not api_key:
                raise RuntimeError(
                    "PIXABAY_API_KEY belum tersedia untuk B-roll Pixabay."
                )

            parsed = urlsplit(url_text)
            url_params = {
                key: values[-1] if values else ""
                for key, values in parse_qs(parsed.query).items()
            }
            call_params = dict(kwargs.get("params") or {})
            merged = {**url_params, **call_params}

            query = str(
                merged.get("query")
                or merged.get("q")
                or ""
            ).strip()

            try:
                per_page = int(merged.get("per_page") or 15)
            except (TypeError, ValueError):
                per_page = 15

            # Pixabay accepts 3..200 results per request.
            per_page = max(3, min(per_page, 200))

            pixabay_params = {
                "key": api_key,
                "q": query[:100],
                "video_type": "all",
                "safesearch": "true",
                "order": "popular",
                "per_page": per_page,
            }

            if not _printed_once["value"]:
                print(
                    "      🎞️ B-roll provider: Pixabay Video API "
                    "(24h search cache enabled)",
                    flush=True,
                )
                _printed_once["value"] = True

            cache_file = _cache_path(pixabay_params)
            cached = _load_cached_payload(cache_file)
            if cached is not None:
                return _synthetic_json_response(
                    cached,
                    "https://pixabay.com/api/videos/",
                )

            forwarded = dict(kwargs)
            forwarded["params"] = pixabay_params

            # Do not forward Pexels Authorization to Pixabay.
            headers = dict(forwarded.get("headers") or {})
            headers.pop("Authorization", None)
            headers.pop("authorization", None)
            if headers:
                forwarded["headers"] = headers
            else:
                forwarded.pop("headers", None)

            response = _original_request(
                self,
                method,
                "https://pixabay.com/api/videos/",
                **forwarded,
            )

            if response.ok:
                try:
                    transformed = _pixabay_payload_to_pexels(response.json())
                    _save_cached_payload(cache_file, transformed)

                    response._content = json.dumps(
                        transformed,
                        ensure_ascii=False,
                    ).encode("utf-8")
                    response.headers["Content-Type"] = "application/json"
                    response.headers["X-OSC-BRoll-Provider"] = "Pixabay"
                    response.encoding = "utf-8"
                except Exception as exc:
                    raise RuntimeError(
                        f"Gagal mengonversi respons Pixabay untuk B-roll: {exc}"
                    ) from exc

            return response

        return _original_request(self, method, url, **kwargs)

    _request_with_pixabay._osc_pixabay_adapter = True
    requests.sessions.Session.request = _request_with_pixabay
"""

launcher_source = r"""
import os

# Must be imported before the FastAPI app so requests is patched before
# clipping/studio/broll.py performs any Pexels API request.
import osc_pixabay_adapter  # noqa: F401
import uvicorn

uvicorn.run(
    "web.api.app:app",
    host="0.0.0.0",
    port=int(os.environ.get("OSC_BACKEND_PORT", "8000")),
    log_level="warning",
    access_log=False,
)
"""

compile(adapter_source, str(adapter_file), "exec")
compile(launcher_source, str(launcher_file), "exec")

adapter_file.write_text(adapter_source, encoding="utf-8")
launcher_file.write_text(launcher_source, encoding="utf-8")

print(f"✅ Pixabay B-roll adapter written: {adapter_file}")
print(f"✅ Patched backend launcher written: {launcher_file}")
print("✅ Pixabay search cache: 24 hours")

# Verify schema conversion and requests patch in a CLEAN interpreter.
check_code = r"""
import os
import requests
import osc_pixabay_adapter

patched = getattr(
    requests.sessions.Session.request,
    "_osc_pixabay_adapter",
    False,
)

print("BROLL_PROVIDER:", os.environ.get("BROLL_PROVIDER"))
print("PIXABAY_ADAPTER_PATCHED:", patched)

if os.environ.get("BROLL_PROVIDER") == "pixabay" and not patched:
    raise RuntimeError("Pixabay adapter tidak aktif.")

sample = {
    "totalHits": 1,
    "hits": [
        {
            "id": 125,
            "pageURL": "https://pixabay.com/videos/id-125/",
            "tags": "flowers, yellow",
            "duration": 12,
            "user": "Pixabay Creator",
            "videos": {
                "medium": {
                    "url": "https://cdn.pixabay.com/example.mp4",
                    "width": 1920,
                    "height": 1080,
                    "size": 123456,
                    "thumbnail": "https://cdn.pixabay.com/example.jpg",
                }
            },
        }
    ],
}

converted = osc_pixabay_adapter._pixabay_payload_to_pexels(sample)
assert converted["videos"][0]["video_files"][0]["link"].endswith(".mp4")
assert converted["videos"][0]["image"].endswith(".jpg")

print("PIXABAY_SCHEMA_ADAPTER_OK")
"""

check = subprocess.run(
    [sys.executable, "-c", check_code],
    cwd=str(PROJECT_DIR),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(check.stdout)

if (
    check.returncode != 0
    or "PIXABAY_SCHEMA_ADAPTER_OK" not in check.stdout
):
    raise RuntimeError(
        "Self-test Pixabay adapter gagal.\n\n" + check.stdout[-3000:]
    )

print("✅ Pixabay B-roll adapter ready")


## 5. YouTube Cookies

Karena IP Google Colab sering terkena `429` / `Sign in to confirm you're not a bot`, notebook ini menggunakan file cookies manual untuk source YouTube.

Jika `/content/youtube_cookies.txt` belum ada, file picker Colab akan muncul. Pilih file `youtube_cookies.txt` Anda.


In [ ]:
import os
from pathlib import Path

if REQUIRE_YOUTUBE_COOKIES:
    if not COOKIE_PATH.exists():
        from google.colab import files

        print("🍪 Upload youtube_cookies.txt...")
        uploaded = files.upload()

        if not uploaded:
            raise RuntimeError(
                "Tidak ada file cookies yang diupload. "
                "Upload youtube_cookies.txt atau ubah REQUIRE_YOUTUBE_COOKIES=False."
            )

        if "youtube_cookies.txt" in uploaded:
            cookie_bytes = uploaded["youtube_cookies.txt"]
        else:
            first_name = next(iter(uploaded))
            cookie_bytes = uploaded[first_name]

        COOKIE_PATH.write_bytes(cookie_bytes)

    raw = COOKIE_PATH.read_text(encoding="utf-8-sig", errors="ignore")
    lines = [line.strip() for line in raw.splitlines() if line.strip()]

    if not lines or not lines[0].startswith("#"):
        raise RuntimeError(
            "Format cookie tidak valid. Gunakan export Netscape cookies.txt."
        )

    header_ok = any(
        "Netscape HTTP Cookie File" in line or "HTTP Cookie File" in line
        for line in lines[:5]
    )
    if not header_ok:
        raise RuntimeError(
            "Header Netscape cookies tidak ditemukan. "
            "Export ulang sebagai cookies.txt/Netscape format."
        )

    youtube_rows = sum(
        1 for line in lines
        if (
            (not line.startswith("#") or line.startswith("#HttpOnly_"))
            and ("youtube.com" in line or "google.com" in line)
        )
    )

    if youtube_rows == 0:
        raise RuntimeError(
            "File cookies tidak memiliki entry YouTube/Google yang terdeteksi."
        )

    os.chmod(COOKIE_PATH, 0o600)
    os.environ["YT_COOKIES_FILE"] = str(COOKIE_PATH)

    print("✅ youtube_cookies.txt valid")
    print(f"  Path          : {COOKIE_PATH}")
    print(f"  Relevant rows : {youtube_rows}")
else:
    os.environ.pop("YT_COOKIES_FILE", None)
    print("ℹ️ YouTube cookies dinonaktifkan.")


## 6. Patch Downloader agar Web Studio Memakai Cookies

In [ ]:
from pathlib import Path

engine_file = PROJECT_DIR / "clipping" / "engine.py"
if not engine_file.exists():
    raise FileNotFoundError(engine_file)

src = engine_file.read_text(encoding="utf-8")
PATCH_SENTINEL = "# OSC_COLAB_COOKIE_PATCH"

if PATCH_SENTINEL in src:
    print("✅ Cookie patch sudah aktif")
else:
    injection = '''
    # OSC_COLAB_COOKIE_PATCH
    _osc_cookie_file = os.environ.get("YT_COOKIES_FILE", "").strip()
    if _osc_cookie_file:
        if os.path.isfile(_osc_cookie_file):
            ydl_opts["cookiefile"] = _osc_cookie_file
            print("      🍪 YouTube cookies aktif.", flush=True)
        else:
            print(
                f"      ⚠️ YT_COOKIES_FILE tidak ditemukan: {_osc_cookie_file}",
                flush=True,
            )

'''

    preferred_marker = "    # --- Subtitle download — only supported for YouTube ---"
    fallback_marker = "    # Jalankan download video terpisah dari urusan subtitle"

    if preferred_marker in src:
        src = src.replace(preferred_marker, injection + preferred_marker, 1)
    elif fallback_marker in src:
        src = src.replace(fallback_marker, injection + fallback_marker, 1)
    else:
        raise RuntimeError(
            "Struktur clipping/engine.py berubah dan lokasi patch cookies "
            "tidak ditemukan. Notebook dihentikan agar source tidak rusak."
        )

    compile(src, str(engine_file), "exec")
    engine_file.write_text(src, encoding="utf-8")
    print("✅ Cookie patch diterapkan ke clipping/engine.py")

patched = engine_file.read_text(encoding="utf-8")
if PATCH_SENTINEL not in patched or 'ydl_opts["cookiefile"]' not in patched:
    raise RuntimeError("Cookie patch gagal diverifikasi.")

print("✅ Downloader Web Studio siap menggunakan YT_COOKIES_FILE")


In [ ]:

# ============================================================
# GEMINI V5 — MULTI-MODEL FAILOVER FOR 503 / MODEL OUTAGES
# ============================================================
import os
import re
from pathlib import Path

PRIMARY_GEMINI_MODEL = "gemini-3.6-flash"
FALLBACK_GEMINI_MODEL = "gemini-3.5-flash-lite"

MODEL_POOL = [
    "gemini-3.6-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
    "gemini-3.7-flash",
]

config_file = PROJECT_DIR / "clipping" / "config.py"
engine_file = PROJECT_DIR / "clipping" / "engine.py"

if not config_file.exists():
    raise FileNotFoundError(config_file)
if not engine_file.exists():
    raise FileNotFoundError(engine_file)

# ------------------------------------------------------------
# 1) Update backend defaults.
# ------------------------------------------------------------
config_src = config_file.read_text(encoding="utf-8")

config_src = re.sub(
    r'(?m)^GEMINI_MODEL\s*=\s*"[^"]+"\s*$',
    f'GEMINI_MODEL = "{PRIMARY_GEMINI_MODEL}"',
    config_src,
)
config_src = re.sub(
    r'(?m)^GEMINI_FALLBACK_MODEL\s*=\s*"[^"]+"\s*$',
    f'GEMINI_FALLBACK_MODEL = "{FALLBACK_GEMINI_MODEL}"',
    config_src,
)

compile(config_src, str(config_file), "exec")
config_file.write_text(config_src, encoding="utf-8")

# ------------------------------------------------------------
# 2) Make the repository's existing retry helper fail fast.
#    The V5 wrapper below handles model rotation instead.
# ------------------------------------------------------------
engine_src = engine_file.read_text(encoding="utf-8")

engine_src = engine_src.replace(
    '"gemini-3-flash-preview"',
    f'"{PRIMARY_GEMINI_MODEL}"',
)
engine_src = engine_src.replace(
    '"gemini-2.5-flash"',
    f'"{FALLBACK_GEMINI_MODEL}"',
)

engine_src = re.sub(
    r"(?m)^MAX_ATTEMPTS\s*=\s*\d+\s*$",
    "MAX_ATTEMPTS = 1",
    engine_src,
)
engine_src = re.sub(
    r"(?m)^INITIAL_WAIT_SECONDS\s*=\s*\d+\s*$",
    "INITIAL_WAIT_SECONDS = 5",
    engine_src,
)
engine_src = re.sub(
    r"(?m)^WAIT_INCREMENT_SECONDS\s*=\s*\d+\s*$",
    "WAIT_INCREMENT_SECONDS = 5",
    engine_src,
)

# Remove the previous V4 injection if it happens to be present in the
# cloned/modified source. The fresh Colab clone normally will not contain it.
v4_sentinel = "    # OSC_GEMINI_2026_MODEL_NORMALIZATION\n"
if v4_sentinel in engine_src:
    start = engine_src.index(v4_sentinel)
    possible_original_body = []
    for marker in (
        "    last_error = None\n",
        "    last_exc = None\n",
        "    for attempt in range(",
    ):
        pos = engine_src.find(marker, start + len(v4_sentinel))
        if pos != -1:
            possible_original_body.append(pos)

    if possible_original_body:
        end = min(possible_original_body)
        engine_src = engine_src[:start] + engine_src[end:]

compile(engine_src, str(engine_file), "exec")
engine_file.write_text(engine_src, encoding="utf-8")

# ------------------------------------------------------------
# 3) Append a wrapper around the repository's JSON helper.
#    Existing analyze_with_gemini() resolves this global at runtime.
# ------------------------------------------------------------
wrapper = r'''
# ============================================================
# OSC GEMINI V5 MULTI-MODEL FAILOVER
# ============================================================
if "_OSC_V5_BASE_GENERATE_JSON" not in globals():
    import inspect as _osc_inspect
    import time as _osc_time

    _OSC_V5_BASE_GENERATE_JSON = _generate_json_with_retry

    _OSC_V5_MODEL_POOL = [
        "gemini-3.6-flash",
        "gemini-3.5-flash-lite",
        "gemini-3.5-flash",
        "gemini-3.1-flash-lite",
        "gemini-3.8-flash",
        "gemini-3.7-flash",
    ]

    _OSC_V5_LEGACY_MODEL_MAP = {
        "gemini-3-flash-preview": "gemini-3.6-flash",
        "gemini-2.5-flash": "gemini-3.5-flash-lite",
    }

    def _osc_v5_unique(items):
        result = []
        for item in items:
            if item and item not in result:
                result.append(item)
        return result

    def _generate_json_with_retry(*args, **kwargs):
        sig = _osc_inspect.signature(_OSC_V5_BASE_GENERATE_JSON)
        bound = sig.bind_partial(*args, **kwargs)
        bound.apply_defaults()

        requested = bound.arguments.get("model")
        requested_fallback = bound.arguments.get("fallback_model")

        requested = _OSC_V5_LEGACY_MODEL_MAP.get(requested, requested)
        requested_fallback = _OSC_V5_LEGACY_MODEL_MAP.get(
            requested_fallback, requested_fallback
        )

        chain = _osc_v5_unique(
            [requested, requested_fallback] + list(_OSC_V5_MODEL_POOL)
        )

        errors = []

        # Two sweeps max. Each base helper is configured for one primary
        # request + one fallback request, with no long backoff.
        for sweep in range(1, 3):
            print(
                f"[Gemini V5] Failover sweep {sweep}/2 "
                f"across {len(chain)} model(s)..."
            )

            for idx in range(0, len(chain), 2):
                primary = chain[idx]
                fallback = (
                    chain[idx + 1]
                    if idx + 1 < len(chain)
                    else chain[idx]
                )

                call_args = dict(bound.arguments)
                call_args["model"] = primary
                call_args["fallback_model"] = fallback

                print(
                    f"[Gemini V5] Trying {primary}"
                    + (f" -> {fallback}" if fallback != primary else "")
                )

                try:
                    result = _OSC_V5_BASE_GENERATE_JSON(**call_args)
                    print(
                        f"[Gemini V5] ✅ Success with "
                        f"{primary}/{fallback}"
                    )
                    return result
                except Exception as exc:
                    msg = str(exc)
                    errors.append(f"{primary}/{fallback}: {msg}")
                    print(
                        f"[Gemini V5] ⚠️ Pair failed: "
                        f"{primary}/{fallback} | {msg[:500]}"
                    )

            if sweep == 1:
                print(
                    "[Gemini V5] Semua model pada sweep pertama gagal. "
                    "Menunggu 12 detik sebelum sweep terakhir..."
                )
                _osc_time.sleep(12)

        summary = " | ".join(errors[-8:])
        raise RuntimeError(
            "Gemini V5: semua model failover gagal setelah 2 sweep. "
            + summary
        )
'''

engine_src = engine_file.read_text(encoding="utf-8")
if "# OSC GEMINI V5 MULTI-MODEL FAILOVER" not in engine_src:
    engine_src = engine_src.rstrip() + "\n\n" + wrapper + "\n"

compile(engine_src, str(engine_file), "exec")
engine_file.write_text(engine_src, encoding="utf-8")

# ------------------------------------------------------------
# 4) Discover model IDs visible to this API key.
#    This is metadata-only; actual job-time 503 is still handled by V5.
# ------------------------------------------------------------
from google import genai

api_key = os.environ.get("GOOGLE_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("GOOGLE_API_KEY tidak tersedia.")

client = genai.Client(api_key=api_key)

visible_models = set()
try:
    for model_info in client.models.list():
        name = getattr(model_info, "name", "") or ""
        visible_models.add(name.removeprefix("models/"))
except Exception as exc:
    print(f"⚠️ models.list gagal; pool default tetap digunakan: {exc}")

if visible_models:
    usable = [m for m in MODEL_POOL if m in visible_models]
    missing = [m for m in MODEL_POOL if m not in visible_models]

    print("✅ Gemini model pool yang terlihat untuk API key ini:")
    for model_name in usable:
        print("   -", model_name)

    if missing:
        print("ℹ️ Model yang tidak terlihat:")
        for model_name in missing:
            print("   -", model_name)

    if usable:
        patched = engine_file.read_text(encoding="utf-8")
        replacement = "_OSC_V5_MODEL_POOL = " + repr(usable)
        patched = re.sub(
            r"_OSC_V5_MODEL_POOL\s*=\s*\[[\s\S]*?\]\n\n    _OSC_V5_LEGACY_MODEL_MAP",
            replacement + "\n\n    _OSC_V5_LEGACY_MODEL_MAP",
            patched,
            count=1,
        )
        compile(patched, str(engine_file), "exec")
        engine_file.write_text(patched, encoding="utf-8")

print("✅ Gemini V5 multi-model failover installed")
print(f"   Default primary : {PRIMARY_GEMINI_MODEL}")
print(f"   Default fallback: {FALLBACK_GEMINI_MODEL}")
print("   Model pool      :", ", ".join(MODEL_POOL))
print("   Policy          : 1 request/model, 2 sweeps, 12s between sweeps")


## 7. Clean-Interpreter Self-Test: NumPy, SciPy, MediaPipe, Studio Imports, dan CUDA

Semua pemeriksaan scientific stack dilakukan di **Python subprocess baru**. Ini mencegah state NumPy/SciPy yang sudah ter-load di kernel Colab mencemari backend.


In [ ]:
import os
import sys
import subprocess
import importlib.metadata as md

os.chdir(PROJECT_DIR)

def run_capture(code_text):
    return subprocess.run(
        [sys.executable, "-c", code_text],
        cwd=str(PROJECT_DIR),
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

# ------------------------------------------------------------
# A. Verify scientific stack + MediaPipe in a CLEAN interpreter.
#    This catches the TensorFlow -> SciPy -> NumPy failure that can happen
#    when Colab's notebook kernel has stale scientific modules loaded.
# ------------------------------------------------------------
render_import_test = r"""
import importlib.util
import numpy as np
import scipy
import mediapipe as mp

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow visible:", importlib.util.find_spec("tensorflow") is not None)

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from clipping import studio, diarization as diarization_mod

print("MEDIAPIPE_RENDER_IMPORT_OK")
"""

render_result = run_capture(render_import_test)

print(render_result.stdout)

if (
    render_result.returncode != 0
    or "MEDIAPIPE_RENDER_IMPORT_OK" not in render_result.stdout
):
    raise RuntimeError(
        "MediaPipe/render import test gagal. Server tidak dijalankan.\n\n"
        + render_result.stdout[-5000:]
    )

# ------------------------------------------------------------
# B. Faster-Whisper FeatureExtractor test in a clean interpreter.
# ------------------------------------------------------------
feature_test = r"""
from faster_whisper.feature_extractor import FeatureExtractor
import numpy as np
import faster_whisper
import ctranslate2

FeatureExtractor()

print("NumPy:", np.__version__)
print("Faster-Whisper:", faster_whisper.__version__)
print("CTranslate2:", ctranslate2.__version__)
print("FEATURE_EXTRACTOR_OK")
"""

feature_result = run_capture(feature_test)

print(feature_result.stdout)

if (
    feature_result.returncode != 0
    or "FEATURE_EXTRACTOR_OK" not in feature_result.stdout
):
    raise RuntimeError(
        "Faster-Whisper FeatureExtractor test gagal. Server tidak dijalankan.\n\n"
        + feature_result.stdout[-5000:]
    )

# ------------------------------------------------------------
# C. CUDA test in a clean interpreter.
# ------------------------------------------------------------
cuda_test = r"""
import ctypes
import os
import torch
import ctranslate2

print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH", ""))

# These are the exact libraries required by current Faster-Whisper GPU
# execution. Loading them explicitly catches the failure that previously
# appeared only when the real transcription job started.
for library in ("libcublas.so.12", "libcudnn.so.9"):
    try:
        ctypes.CDLL(library)
        print(f"{library}: OK")
    except OSError as exc:
        raise RuntimeError(f"{library} tidak dapat dimuat: {exc}") from exc

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia.")

print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

supported = ctranslate2.get_supported_compute_types("cuda")
print("CTranslate2 CUDA compute types:", sorted(supported))

if "float16" not in supported:
    raise RuntimeError("float16 tidak didukung pada GPU ini.")

print("CUDA_STACK_OK")
"""

cuda_result = run_capture(cuda_test)

print(cuda_result.stdout)

if cuda_result.returncode != 0 or "CUDA_STACK_OK" not in cuda_result.stdout:
    raise RuntimeError(
        "CUDA self-test gagal. Server tidak dijalankan.\n\n"
        + cuda_result.stdout[-5000:]
    )

packages = [
    "numpy",
    "scipy",
    "mediapipe",
    "yt-dlp",
    "faster-whisper",
    "ctranslate2",
    "av",
    "fastapi",
    "uvicorn",
    "pyngrok",
]

print("\nInstalled package versions:")
for package in packages:
    try:
        print(f"  {package:<18} {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"  {package:<18} NOT FOUND")

print("\n✅ Scientific stack + MediaPipe + CUDA self-test passed")


## 8. PyAV Audio Decode + Whisper CUDA Smoke Test

Cell ini menguji **dua jalur sebelum server dibuka**:

1. `faster_whisper.audio.decode_audio()` dengan file WAV sementara — ini menangkap konflik PyAV seperti `metadata_errors`.
2. Menjalankan **transkripsi nyata** dengan Whisper `tiny` di CUDA — ini memaksa CTranslate2 memuat cuBLAS/cuDNN dan menjalankan encoder sebelum backend dibuka.

Model yang dipilih di Studio (misalnya `large-v3`) tetap digunakan untuk job sebenarnya.


In [ ]:
import os
import sys
import subprocess

# ------------------------------------------------------------
# A. Verify the exact PyAV API Faster-Whisper 1.2.1 requires.
#    PyAV 19 removed metadata_errors and breaks decode_audio().
# ------------------------------------------------------------
av_test_code = r"""
import tempfile
import wave
import struct
import os
import av
from faster_whisper.audio import decode_audio

print("PyAV:", av.__version__)

if tuple(map(int, av.__version__.split(".")[:2])) >= (19, 0):
    raise RuntimeError(
        "PyAV 19+ tidak kompatibel dengan Faster-Whisper 1.2.1 "
        "karena metadata_errors dihapus."
    )

fd, wav_path = tempfile.mkstemp(suffix=".wav")
os.close(fd)

try:
    sample_rate = 16000
    duration_seconds = 1
    samples = [0] * (sample_rate * duration_seconds)

    with wave.open(wav_path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(b"".join(struct.pack("<h", s) for s in samples))

    audio = decode_audio(wav_path, sampling_rate=16000)

    if audio is None or len(audio) == 0:
        raise RuntimeError("decode_audio menghasilkan audio kosong.")

    print("PYAV_DECODE_OK")
finally:
    try:
        os.remove(wav_path)
    except OSError:
        pass
"""

print("⏳ PyAV/Faster-Whisper audio decode smoke test...")
av_test = subprocess.run(
    [sys.executable, "-c", av_test_code],
    cwd=str(PROJECT_DIR),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(av_test.stdout)

if av_test.returncode != 0 or "PYAV_DECODE_OK" not in av_test.stdout:
    raise RuntimeError(
        "PyAV/Faster-Whisper audio decode test gagal. "
        "Server tidak dijalankan karena transkripsi pasti gagal."
    )

# ------------------------------------------------------------
# B. Verify Faster-Whisper can perform REAL CUDA inference.
#    Loading WhisperModel alone is not enough: cuBLAS is first required when
#    CTranslate2 actually runs the encoder. This test transcribes a tiny WAV
#    so libcublas.so.12/libcudnn.so.9 are exercised before the server starts.
# ------------------------------------------------------------
smoke_code = f"""
import os
import struct
import tempfile
import wave

from faster_whisper import WhisperModel

fd, wav_path = tempfile.mkstemp(suffix=".wav")
os.close(fd)

try:
    sample_rate = 16000
    duration_seconds = 1

    # Low-amplitude synthetic tone. It is deterministic and forces audio
    # through the Whisper encoder without needing any external media.
    frames = []
    import math
    for i in range(sample_rate * duration_seconds):
        sample = int(800 * math.sin(2 * math.pi * 440 * i / sample_rate))
        frames.append(struct.pack("<h", sample))

    with wave.open(wav_path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(b"".join(frames))

    model = WhisperModel(
        "{WHISPER_SMOKE_MODEL}",
        device="cuda",
        compute_type="{WHISPER_COMPUTE_TYPE}",
    )

    segments, info = model.transcribe(
        wav_path,
        language="en",
        beam_size=1,
        word_timestamps=False,
    )

    # Faster-Whisper segments are lazy. Iterating is essential because this is
    # what triggers the actual CTranslate2 CUDA encoder/decoder execution.
    list(segments)

    print("WHISPER_CUDA_INFERENCE_OK")
finally:
    try:
        os.remove(wav_path)
    except OSError:
        pass
"""

print(f"⏳ Whisper CUDA inference smoke test ({WHISPER_SMOKE_MODEL})...")
smoke = subprocess.run(
    [sys.executable, "-c", smoke_code],
    cwd=str(PROJECT_DIR),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(smoke.stdout)

if smoke.returncode != 0 or "WHISPER_CUDA_INFERENCE_OK" not in smoke.stdout:
    raise RuntimeError(
        "Whisper CUDA inference smoke test gagal. Server tidak dijalankan "
        "karena transkripsi nyata juga akan gagal.\n\n"
        + smoke.stdout[-5000:]
    )

print("✅ PyAV audio decode ready")
print("✅ Faster-Whisper real CUDA inference ready")


## 9. Start Backend + ngrok

**Cell terakhir akan terus berjalan. Jangan dihentikan selama memakai Studio.**

Cell ini sekarang melakukan tiga pemeriksaan sebelum menyatakan backend siap:

1. FastAPI membuka port lokal.
2. `/api/health` dapat diakses melalui URL publik ngrok.
3. Response ngrok mengandung header CORS yang mengizinkan `https://raymanq.github.io`.

Setelah muncul **Public URL**, buka:

**https://raymanq.github.io/opensource-clipping/studio/**

Lalu **Connect → paste Public URL → masukkan Backend Access Token jika `OSC_API_TOKEN` digunakan → Test & Connect**.


In [ ]:
import os
import sys
import time
import socket
import subprocess
from pathlib import Path

import requests

os.chdir(PROJECT_DIR)

required_env = ["GOOGLE_API_KEY", "NGROK_AUTHTOKEN", "OSC_ALLOWED_ORIGINS"]
missing_env = [name for name in required_env if not os.environ.get(name)]
if missing_env:
    raise RuntimeError("Environment wajib hilang: " + ", ".join(missing_env))

if REQUIRE_YOUTUBE_COOKIES:
    cookie = Path(os.environ.get("YT_COOKIES_FILE", ""))
    if not cookie.is_file():
        raise RuntimeError("YT_COOKIES_FILE tidak tersedia.")

from pyngrok import ngrok

# Kill stale ngrok tunnels from a prior cell run.
try:
    ngrok.kill()
except Exception:
    pass

# Kill a stale uvicorn child from this notebook, if any.
old_pid_file = Path("/content/osc_uvicorn.pid")
if old_pid_file.exists():
    try:
        old_pid = int(old_pid_file.read_text().strip())
        os.kill(old_pid, 15)
        time.sleep(1)
    except Exception:
        pass
    old_pid_file.unlink(missing_ok=True)

server_env = os.environ.copy()
server_env["PYTHONUNBUFFERED"] = "1"
server_env["OSC_BACKEND_PORT"] = str(BACKEND_PORT)

server_proc = subprocess.Popen(
    [
        sys.executable,
        "osc_server.py",
    ],
    cwd=str(PROJECT_DIR),
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

old_pid_file.write_text(str(server_proc.pid), encoding="utf-8")

def wait_for_port(host, port, process, timeout=90):
    deadline = time.time() + timeout

    while time.time() < deadline:
        if process.poll() is not None:
            remaining = process.stdout.read() if process.stdout else ""
            raise RuntimeError(
                "Backend berhenti saat startup.\n\n" + remaining[-5000:]
            )

        try:
            with socket.create_connection((host, port), timeout=1):
                return
        except OSError:
            time.sleep(0.5)

    raise TimeoutError(
        f"Backend tidak membuka port {port} dalam {timeout} detik."
    )

def wait_for_http(url, *, headers=None, process=None, timeout=60):
    deadline = time.time() + timeout
    last_error = None

    while time.time() < deadline:
        if process is not None and process.poll() is not None:
            remaining = process.stdout.read() if process.stdout else ""
            raise RuntimeError(
                "Backend berhenti ketika health check dijalankan.\n\n"
                + remaining[-5000:]
            )

        try:
            response = requests.get(
                url,
                headers=headers or {},
                timeout=10,
                allow_redirects=True,
            )
            if response.status_code < 500:
                return response
            last_error = RuntimeError(
                f"HTTP {response.status_code}: {response.text[:500]}"
            )
        except Exception as exc:
            last_error = exc

        time.sleep(1.0)

    raise RuntimeError(
        f"Health check gagal untuk {url}. Error terakhir: {last_error}"
    )

print("🚀 Starting OpenSource Clipping backend in clean subprocess...")
print(f"🎞️ B-roll provider: {os.environ.get('BROLL_PROVIDER', 'pexels')}")
print(f"🌐 Allowed origin : {os.environ.get('OSC_ALLOWED_ORIGINS')}")
print(
    "🔐 API token      : "
    + ("enabled" if os.environ.get("OSC_API_TOKEN") else "disabled")
)

wait_for_port("127.0.0.1", BACKEND_PORT, server_proc)

# Verify the backend itself before opening an internet tunnel.
local_health_url = f"http://127.0.0.1:{BACKEND_PORT}/api/health"
local_health = wait_for_http(
    local_health_url,
    process=server_proc,
    timeout=30,
)
if not local_health.ok:
    raise RuntimeError(
        f"Local health check gagal: HTTP {local_health.status_code} "
        f"{local_health.text[:1000]}"
    )
print("✅ Local FastAPI health check passed")

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
tunnel = ngrok.connect(addr=BACKEND_PORT, proto="http", bind_tls=True)
public_url = tunnel.public_url.rstrip("/")

Path("/content/OSC_BACKEND_URL.txt").write_text(
    public_url + "\n",
    encoding="utf-8",
)

# Verify the exact request pattern used by a browser hosted on GitHub Pages.
public_health_url = public_url + "/api/health"
cors_headers = {
    "Origin": STUDIO_ORIGIN,
    "ngrok-skip-browser-warning": "true",
}
public_health = wait_for_http(
    public_health_url,
    headers=cors_headers,
    process=server_proc,
    timeout=60,
)

if not public_health.ok:
    raise RuntimeError(
        f"Public health check gagal: HTTP {public_health.status_code} "
        f"{public_health.text[:1000]}"
    )

allow_origin = (
    public_health.headers.get("access-control-allow-origin", "")
    .strip()
    .rstrip("/")
)

if allow_origin not in {STUDIO_ORIGIN, "*"}:
    raise RuntimeError(
        "Backend online, tetapi CORS belum mengizinkan GitHub Pages.\n"
        f"Origin yang diuji : {STUDIO_ORIGIN}\n"
        f"Header dari server: {allow_origin or '(tidak ada)'}\n"
        "Pastikan Colab Secret OSC_ALLOWED_ORIGINS bernilai persis "
        f"{STUDIO_ORIGIN}, lalu jalankan ulang cell Secrets dan cell backend."
    )

print("✅ Public ngrok health check passed")
print(f"✅ CORS confirmed for {STUDIO_ORIGIN}")

print()
print("╔" + "═" * 76 + "╗")
print("║" + " ✅ OPENSOURCE CLIPPING BACKEND READY".ljust(76) + "║")
print("║" + "".ljust(76) + "║")
print("║" + f" Public URL : {public_url}".ljust(76) + "║")
print("║" + f" Health     : {public_health_url}".ljust(76) + "║")
print("║" + f" Studio     : {STUDIO_URL}".ljust(76) + "║")
print("║" + f" CORS       : {STUDIO_ORIGIN} ✅".ljust(76) + "║")
print(
    "║"
    + (
        " API Token  : enabled — use the same token in Backend Access Token"
        if os.environ.get("OSC_API_TOKEN")
        else " API Token  : disabled"
    ).ljust(76)
    + "║"
)
print("║" + "".ljust(76) + "║")
print("║" + " Copy Public URL → Studio → Connect → Test & Connect".ljust(76) + "║")
print("╚" + "═" * 76 + "╝")
print()
print("⚠️ SERVER RUNNING — DO NOT STOP THIS CELL WHILE USING STUDIO")
print()

try:
    assert server_proc.stdout is not None
    for line in iter(server_proc.stdout.readline, ""):
        print(line, end="", flush=True)

        if server_proc.poll() is not None:
            break

    rc = server_proc.wait()

    if rc != 0:
        raise RuntimeError(f"Backend exited with code {rc}")

except KeyboardInterrupt:
    print("\n🛑 Stopping backend...")
    try:
        server_proc.terminate()
        server_proc.wait(timeout=10)
    except Exception:
        try:
            server_proc.kill()
        except Exception:
            pass

    try:
        ngrok.kill()
    except Exception:
        pass

    raise
